In [ ]:
def main(datasources, start_date, end_date):
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]
    sql = f"""

    WITH raw AS (
        SELECT date, instrument, CAST(date AS DATE) AS d,
            CAST(EXTRACT(HOUR FROM date) AS INTEGER) * 100
                + CAST(EXTRACT(MINUTE FROM date) AS INTEGER) AS hhmm,
            CAST(EXTRACT(HOUR FROM date) AS INTEGER) * 60
                + CAST(EXTRACT(MINUTE FROM date) AS INTEGER) AS minute_of_day,
            amount,
            close,
            volume
        FROM {bar1m}
    ),
    tagged AS (
        SELECT *, CASE
            WHEN hhmm BETWEEN 931 AND 1130
                THEN CAST(FLOOR((minute_of_day - 571) / 30.0) AS INTEGER)
            WHEN hhmm BETWEEN 1301 AND 1500
                THEN 4 + CAST(FLOOR((minute_of_day - 781) / 30.0) AS INTEGER)
        END AS block_id
        FROM raw
        WHERE (hhmm BETWEEN 931 AND 1130) OR (hhmm BETWEEN 1301 AND 1500)
    )
    SELECT d, instrument, block_id, MAX(date) AS bar_ts,
        SUM(CAST(amount AS DOUBLE)) AS amount,
        ARG_MAX(CAST(close AS DOUBLE), date) AS close,
        SUM(CAST(volume AS DOUBLE)) AS volume
    FROM tagged
    GROUP BY d, instrument, block_id
    ORDER BY instrument, bar_ts
    
    """

    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    if (end.hour, end.minute, end.second) == (0, 0, 0):
        end = end + pd.Timedelta(hours=23, minutes=59, seconds=59)
    filters = {"date": [str(start), str(end)]}
    bars = dai.query(sql, filters=filters, compression=True).df()
    bars["bar_ts"] = pd.to_datetime(bars["bar_ts"])
    bars["d"] = pd.to_datetime(bars["d"])
    bars["instrument"] = bars["instrument"].astype(str)
    bars = bars.sort_values(["instrument", "bar_ts"]).reset_index(drop=True)

    key = ["d", "instrument"]
    g = bars.groupby(key, sort=False)
    last_idx = g.tail(1).set_index(key)
    day = pd.DataFrame(index=last_idx.index)
    am = bars.loc[bars["block_id"] <= 3].groupby(key, sort=False)["volume"].sum()
    day["amount"] = g["amount"].sum()
    day["l30_amount"] = last_idx["amount"]
    day["close"] = g["close"].last()
    day["l30_close"] = last_idx["close"]
    day["volume"] = g["volume"].sum()
    day["f30_volume"] = g["volume"].first()
    day["l30_volume"] = last_idx["volume"]
    day["am_volume"] = am
    day["amount_last30m_share"] = day["l30_amount"] / day["amount"].replace(0, np.nan)
    _vwap = day["amount"] / day["volume"].replace(0, np.nan)
    day["ret_close_vwap"] = day["close"] / _vwap.replace(0, np.nan) - 1
    day["volume_last30m_share"] = day["l30_volume"] / day["volume"].replace(0, np.nan)
    day = day.reset_index()

    blocks = [('product', 'amount_last30m_share', 'ret_close_vwap', -1.0), ('difference', 'volume_last30m_share', 'amount_last30m_share', -1.0)]
    combine = 'rank'

    def raw_component(op, left, right):
        a = pd.to_numeric(day[left], errors="coerce")
        if op == "raw":
            return a
        if op == "pct_change":
            return a.groupby(day["instrument"]).pct_change(fill_method=None)
        b = pd.to_numeric(day[right], errors="coerce")
        if op == "ratio":
            return a / b.replace(0, np.nan)
        if op == "difference":
            return a - b
        if op == "product":
            return a * b
        raise ValueError("unsupported factor block: " + op)

    normalized = []
    for op, left, right, sign in blocks:
        values = raw_component(op, left, right) * sign
        values = values.replace([np.inf, -np.inf], np.nan)
        if combine == "rank":
            normalized.append(values.groupby(day["d"]).rank(pct=True) - 0.5)
        else:
            normalized.append(values.groupby(day["d"]).transform(
                lambda x: (x - x.mean()) / (x.std(ddof=0) + 1e-9)))
    signal = pd.concat(normalized, axis=1).mean(axis=1, skipna=False)
    if combine == "rank":
        day["factor"] = signal.groupby(day["d"]).rank(pct=True)
    else:
        day["factor"] = signal.groupby(day["d"]).transform(
            lambda x: (x - x.mean()) / (x.std(ddof=0) + 1e-9))
    result = day[["d", "instrument", "factor"]].rename(columns={"d": "date"})

    pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters=filters, compression=True,
    ).df()
    pool["date"] = pd.to_datetime(pool["date"])
    pool["instrument"] = pool["instrument"].astype(str)
    result = pool.merge(result, how="left", on=["date", "instrument"])
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce").replace(
        [np.inf, -np.inf], np.nan).fillna(0.0)
    result = result[["date", "instrument", "factor"]]
    result = result.drop_duplicates(["date", "instrument"], keep="last")
    return result.sort_values(["date", "instrument"]).reset_index(drop=True)
